# Chapitre 2 - Cours 2 : normalisation, jointures et relations plusieurs-à-plusieurs

Le cours 1 a montré le point fort du modèle document : un arbre de relations
un-à-plusieurs lu d'un seul bloc. Ce cours en montre la limite : dès que les
enregistrements **se référencent** entre eux, il faut des identifiants, des jointures, et
choisir entre données **normalisées** et **dénormalisées**. Il couvre les sections
*Many-to-One and Many-to-Many Relationships* et *Are Document Databases Repeating
History?* (*The network model*, *The relational model*, *Comparison to document
databases*).

**Plan.**

1. Un identifiant ou un texte ? La normalisation.
2. Le compromis lecture/écriture et la cohérence des copies.
3. Relations plusieurs-à-un et plusieurs-à-plusieurs.
4. Les bases documents répètent-elles l'histoire ? Modèles hiérarchique et réseau.

In [1]:
import json
import sqlite3
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import datamodels as dm
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET, COLORS,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 90)
rng = np.random.default_rng(0)
fmt = lambda v: f"{v:,.0f}".replace(",", " ")

orgs = dm.make_orgs(200, seed=0)
profiles = dm.make_profiles(20_000, seed=0, orgs=orgs)
con = sqlite3.connect(":memory:")
dm.create_relational(con, profiles, orgs)
dm.create_documents(con, profiles)

## 1. Un identifiant ou un texte ? La normalisation (section *Many-to-One and Many-to-Many Relationships*)

Dans nos profils, la région n'est pas stockée en clair (« Lyon, Auvergne-Rhône-Alpes,
France ») mais par un **identifiant** (`"fr:69"`) qui renvoie à une liste normalisée de
régions. Pourquoi ? Proposer une liste déroulante plutôt qu'un champ libre apporte :

- une **orthographe cohérente** d'un profil à l'autre ;
- l'absence d'**ambiguïté** entre lieux homonymes (Valence en France ou en Espagne ?) ;
- une **mise à jour facile** : le nom n'est stocké qu'à un endroit ;
- la **traduction** de l'interface : la liste se localise, pas les profils ;
- une **meilleure recherche** : la liste peut savoir que Lyon est dans le sud-est de la
  France, ce que la chaîne « Lyon » seule ne dit pas.

Stocker un identifiant, c'est **normaliser** : l'information lisible par un humain n'existe
qu'en un exemplaire, tout le reste y renvoie par un identifiant qui n'a de sens que dans
la base. Stocker le texte dans chaque enregistrement, c'est **dénormaliser** : on
duplique l'information. L'identifiant a un avantage décisif : n'ayant **aucun sens pour
un humain**, il n'a jamais besoin de changer, même quand ce qu'il désigne change.

Construisons les deux variantes de la table des profils et mesurons ce que coûte un
changement de nom de région (par exemple après une fusion de régions).

In [2]:
con.executescript("""
DROP TABLE IF EXISTS users_denorm;
CREATE TABLE users_denorm AS
  SELECT u.id, u.first_name, u.last_name, u.headline, r.name AS region_name
    FROM users u JOIN regions r ON r.id = u.region_id;
""")
taille = lambda t, cols: con.execute(f"SELECT SUM({' + '.join(f'LENGTH({c})' for c in cols)}) FROM {t}").fetchone()[0]
n_lyon = con.execute("SELECT COUNT(*) FROM users WHERE region_id = 'fr:69'").fetchone()[0]

def rename(normalise):
    t0 = time.perf_counter()
    if normalise:
        cur = con.execute("UPDATE regions SET name = 'Lyon, Région Lyonnaise, France' WHERE id = 'fr:69'")
    else:
        cur = con.execute("UPDATE users_denorm SET region_name = 'Lyon, Région Lyonnaise, France' "
                          "WHERE region_name = 'Lyon, Auvergne-Rhône-Alpes, France'")
    con.rollback()                                   # on annule pour garder les données intactes
    return cur.rowcount, (time.perf_counter() - t0) * 1e3

(n1, t1), (n2, t2) = rename(True), rename(False)
pd.DataFrame({
    "représentation": ["normalisée (region_id)", "dénormalisée (texte)"],
    "octets pour la région": [taille("users", ["region_id"]) + taille("regions", ["id", "name"]),
                              taille("users_denorm", ["region_name"])],
    "lignes modifiées par le renommage": [n1, n2],
    "temps (ms)": [t1, t2],
})

,représentation,octets pour la région,lignes modifiées par le renommage,temps (ms)
0,normalisée (region_id),102086,1,0.030
1,dénormalisée (texte),551644,1671,1.539


Le renommage touche **une ligne** dans la version normalisée, et les quelque
1 700 profils lyonnais dans la version dénormalisée, qui occupe aussi plusieurs fois plus de place.
Le prix de la normalisation se paie à la lecture : afficher un profil lisible demande
une **jointure** pour résoudre l'identifiant en texte.

In [3]:
SQL_JOIN = """SELECT users.first_name, users.last_name, regions.name
                FROM users JOIN regions ON users.region_id = regions.id
               WHERE users.id = ?"""
print(con.execute(SQL_JOIN, (251,)).fetchone())
for row in con.execute("EXPLAIN QUERY PLAN " + SQL_JOIN, (251,)):
    print("  plan :", row[-1])

('Yanis', 'Michel', 'Nantes, Pays de la Loire, France')
  plan : SEARCH users USING INTEGER PRIMARY KEY (rowid=?)
  plan : SEARCH regions USING INDEX sqlite_autoindex_regions_1 (id=?)


Les bases documents savent stocker des données normalisées ou non, mais on les associe
souvent à la dénormalisation : JSON rend facile l'ajout de champs redondants, et beaucoup
de bases documents gèrent mal les jointures. Sans jointure côté base, l'application la
fait elle-même : lire le document, puis lancer une seconde requête pour résoudre
l'identifiant. MongoDB propose aussi l'opérateur `$lookup` dans son pipeline
d'agrégation ; notre mini-moteur `dm.aggregate` l'implémente.

In [4]:
docs = [json.loads(r[0]) for r in con.execute("SELECT doc FROM profiles WHERE user_id IN (251, 252)")]
regions_coll = [{"_id": k, "name": v} for k, v in dm.REGIONS.items()]
res = dm.aggregate(docs, [
    {"$match": {"user_id": 251}},
    {"$lookup": {"from": "regions", "localField": "region_id", "foreignField": "_id", "as": "region"}},
    {"$project": {"_id": 0, "first_name": 1, "last_name": 1, "region": 1}},
], collections={"regions": regions_coll})
res

[{'first_name': 'Yanis',
  'last_name': 'Michel',
  'region': [{'_id': 'fr:44', 'name': 'Nantes, Pays de la Loire, France'}]}]

> **À retenir.** Normaliser, c'est ranger l'information lisible en un seul exemplaire et
> la référencer par un identifiant sans signification, qui n'a jamais à changer.
> Dénormaliser, c'est dupliquer le texte : lecture sans jointure, mais mises à jour
> multiples et risque d'incohérence.